In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Notebook 3

# Model Architecture

Knowledge-Guided Chest X-ray Report Generation

This notebook implements the visual encoder.

Pipeline

Chest X-ray
↓

MobileNetV3

↓

Dynamic MDConv

↓

Hybrid Attention

↓

AMSDA

↓

Encoded Visual Features


In [1]:
import os
import cv2
import math
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models
from torchvision.models import mobilenet_v3_large

warnings.filterwarnings("ignore")

print("Libraries Loaded!")

Libraries Loaded!


In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 224

FEATURE_DIM = 960

print("Device :", DEVICE)

Device : cuda


In [5]:
CONFIG = {

    "image_size":224,

    "feature_dim":960,

    "dropout":0.2,

    "attention_channels":960

}

CONFIG

{'image_size': 224,
 'feature_dim': 960,
 'dropout': 0.2,
 'attention_channels': 960}

In [6]:
class MobileNetV3Backbone(nn.Module):

    def __init__(self):

        super().__init__()

        model = mobilenet_v3_large(weights="DEFAULT")

        # Remove classifier
        self.features = model.features

    def forward(self, x):

        x = self.features(x)

        return x

In [7]:
backbone = MobileNetV3Backbone().to(DEVICE)

dummy = torch.randn(
    2,
    3,
    IMG_SIZE,
    IMG_SIZE
).to(DEVICE)

output = backbone(dummy)

print(output.shape)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 211MB/s]


torch.Size([2, 960, 7, 7])


In [8]:
print("="*50)

print("Input Shape :", dummy.shape)

print("Output Shape :", output.shape)

print("="*50)

Input Shape : torch.Size([2, 3, 224, 224])
Output Shape : torch.Size([2, 960, 7, 7])


## Dynamic Multi-Kernel Depthwise Convolution (MDConv)

Purpose:

Extract image features at multiple spatial scales.

Three depthwise convolutions are used:

- 3×3
- 5×5
- 7×7

The outputs are fused using learnable adaptive weights.


In [9]:
class DynamicMDConv(nn.Module):

    def __init__(self,
                 channels):

        super().__init__()

        # 3×3 depthwise
        self.dw3 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            groups=channels
        )

        # 5×5 depthwise
        self.dw5 = nn.Conv2d(
            channels,
            channels,
            kernel_size=5,
            padding=2,
            groups=channels
        )

        # 7×7 depthwise
        self.dw7 = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels
        )

        # Learnable weights
        self.weights = nn.Parameter(
            torch.ones(3)
        )

    def forward(self,x):

        f3 = self.dw3(x)

        f5 = self.dw5(x)

        f7 = self.dw7(x)

        w = torch.softmax(
            self.weights,
            dim=0
        )

        out = (
            w[0]*f3 +
            w[1]*f5 +
            w[2]*f7
        )

        return out

In [10]:
mdconv = DynamicMDConv(
    FEATURE_DIM
).to(DEVICE)

output_md = mdconv(output)

print(output_md.shape)

torch.Size([2, 960, 7, 7])


In [11]:
print("="*50)

print("Before MDConv :", output.shape)

print("After MDConv :", output_md.shape)

print("="*50)

Before MDConv : torch.Size([2, 960, 7, 7])
After MDConv : torch.Size([2, 960, 7, 7])


# Feature Pyramid Network (FPN)

## Purpose

The Feature Pyramid Network combines features from multiple scales to preserve both fine details and high-level semantic information.

It prepares richer feature maps before applying the Hybrid Attention module.

In [17]:
class FeaturePyramidNetwork(nn.Module):

    def __init__(self, channels):

        super().__init__()

        # Reduce channels
        self.lateral = nn.Conv2d(
            channels,
            channels,
            kernel_size=1
        )

        # Feature refinement
        self.smooth = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):

        lateral = self.lateral(x)

        output = self.smooth(lateral)

        return output

In [18]:
fpn = FeaturePyramidNetwork(
    FEATURE_DIM
).to(DEVICE)

output_fpn = fpn(output_md)

print(output_fpn.shape)

torch.Size([2, 960, 7, 7])


In [19]:
print("=" * 50)

print("MDConv Output :", output_md.shape)

print("FPN Output    :", output_fpn.shape)

print("=" * 50)

MDConv Output : torch.Size([2, 960, 7, 7])
FPN Output    : torch.Size([2, 960, 7, 7])


# Hybrid Attention

## Purpose

Hybrid Attention enhances the feature maps by focusing on:

- Important feature channels (Channel Attention)
- Important spatial regions (Spatial Attention)

This helps the model focus on clinically significant structures before AMSDA.

In [20]:
class ChannelAttention(nn.Module):

    def __init__(self, channels, reduction=16):

        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(

            nn.Conv2d(
                channels,
                channels // reduction,
                kernel_size=1,
                bias=False
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels // reduction,
                channels,
                kernel_size=1,
                bias=False
            )

        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = self.fc(self.avg_pool(x))

        max_ = self.fc(self.max_pool(x))

        attention = self.sigmoid(avg + max_)

        return x * attention

In [21]:
class SpatialAttention(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        concat = torch.cat(
            [avg, max_],
            dim=1
        )

        attention = self.sigmoid(
            self.conv(concat)
        )

        return x * attention

In [22]:
class HybridAttention(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.channel_attention = ChannelAttention(
            channels
        )

        self.spatial_attention = SpatialAttention()

    def forward(self, x):

        x = self.channel_attention(x)

        x = self.spatial_attention(x)

        return x

In [23]:
hybrid_attention = HybridAttention(
    FEATURE_DIM
).to(DEVICE)

output_attention = hybrid_attention(output_fpn)

print(output_attention.shape)

torch.Size([2, 960, 7, 7])


In [24]:
print("=" * 50)

print("FPN Output             :", output_fpn.shape)

print("Hybrid Attention Output:", output_attention.shape)

print("=" * 50)

FPN Output             : torch.Size([2, 960, 7, 7])
Hybrid Attention Output: torch.Size([2, 960, 7, 7])


# Feature Projection

Purpose

Project MobileNetV3 features into the feature size required by AMSDA.

960 × 7 × 7

↓

256 × 14 × 14

In [25]:
class FeatureProjection(nn.Module):

    def __init__(self):

        super().__init__()

        self.project = nn.Sequential(

            nn.Conv2d(
                960,
                256,
                kernel_size=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True)

        )

    def forward(self,x):

        x = self.project(x)

        x = F.interpolate(
            x,
            size=(14,14),
            mode="bilinear",
            align_corners=False
        )

        return x

In [26]:
projection = FeatureProjection().to(DEVICE)

amsda_input = projection(output_attention)

print(amsda_input.shape)

torch.Size([2, 256, 14, 14])


In [27]:
print("="*50)

print("Hybrid Output :",output_attention.shape)

print("AMSDA Input   :",amsda_input.shape)

print("="*50)

Hybrid Output : torch.Size([2, 960, 7, 7])
AMSDA Input   : torch.Size([2, 256, 14, 14])


# AMSDA (Adaptive Multi-Scale Denoising Attention)

Purpose

This module extracts fine, medium and global visual information simultaneously.

The three branches are fused using adaptive learnable weights.

A frequency-domain denoising block suppresses noisy features before adding a residual connection.

In [28]:
class MultiScaleBranch(nn.Module):

    def __init__(self, in_channels=256):

        super().__init__()

        self.reduce = nn.Conv2d(
            in_channels,
            64,
            kernel_size=1
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=64,
            num_heads=8,
            batch_first=True
        )

    def forward(self,x):

        x = self.reduce(x)

        B,C,H,W = x.shape

        tokens = x.flatten(2).transpose(1,2)

        tokens,_ = self.attention(
            tokens,
            tokens,
            tokens
        )

        out = tokens.transpose(1,2).reshape(
            B,
            C,
            H,
            W
        )

        return out

In [30]:
class AMSDA(nn.Module):

    def __init__(self):

        super().__init__()

        # Fine-scale branch
        self.fine = MultiScaleBranch()

        # Medium-scale branch
        self.medium = MultiScaleBranch()

        # Global-scale branch
        self.global_conv = nn.Conv2d(
            256,
            64,
            kernel_size=1
        )

        # Adaptive Scale Weighting
        self.weight_predictor = nn.Sequential(

            nn.Linear(192, 64),

            nn.ReLU(),

            nn.Linear(64, 3)

        )

        # Learnable FFT Mask
        self.mask = nn.Parameter(
            torch.ones(64, 14, 14)
        )

        # Project back to 256 channels
        self.project = nn.Conv2d(
            64,
            256,
            kernel_size=1
        )

    def forward(self, x):

        # -------------------------
        # Fine Scale Branch
        # -------------------------
        fine = self.fine(x)

        # -------------------------
        # Medium Scale Branch
        # -------------------------
        medium = F.avg_pool2d(x, 2)

        medium = self.medium(medium)

        medium = F.interpolate(
            medium,
            size=(14, 14),
            mode="bilinear",
            align_corners=False
        )

        # -------------------------
        # Global Scale Branch
        # -------------------------
        global_feat = F.adaptive_avg_pool2d(
            x,
            1
        )

        global_feat = self.global_conv(
            global_feat
        )

        global_feat = global_feat.expand(
            -1,
            -1,
            14,
            14
        )

        # -------------------------
        # Adaptive Scale Weighting
        # -------------------------
        cat = torch.cat([

            F.adaptive_avg_pool2d(fine, 1).flatten(1),

            F.adaptive_avg_pool2d(medium, 1).flatten(1),

            F.adaptive_avg_pool2d(global_feat, 1).flatten(1)

        ], dim=1)

        weights = torch.softmax(

            self.weight_predictor(cat),

            dim=1

        )

        fused = (

            weights[:, 0].view(-1, 1, 1, 1) * fine +

            weights[:, 1].view(-1, 1, 1, 1) * medium +

            weights[:, 2].view(-1, 1, 1, 1) * global_feat

        )

        # -------------------------
        # Frequency-Domain Denoising
        # -------------------------
        fft = torch.fft.fft2(fused)

        mask = torch.sigmoid(self.mask)

        denoised = torch.fft.ifft2(
            fft * mask
        ).real

        # -------------------------
        # Projection + Residual
        # -------------------------
        out = self.project(
            denoised
        )

        out = out + x

        return out

In [31]:
amsda = AMSDA().to(DEVICE)

output_amsda = amsda(amsda_input)

print(output_amsda.shape)

torch.Size([2, 256, 14, 14])


In [32]:
print("="*50)

print("AMSDA Input  :", amsda_input.shape)

print("AMSDA Output :", output_amsda.shape)

print("="*50)

AMSDA Input  : torch.Size([2, 256, 14, 14])
AMSDA Output : torch.Size([2, 256, 14, 14])


# Complete Visual Encoder

This module combines all previously implemented encoder blocks into a single network.

Pipeline

Input Image

↓

MobileNetV3

↓

Dynamic MDConv

↓

Feature Pyramid Network

↓

Hybrid Attention

↓

Feature Projection

↓

AMSDA

↓

Encoded Visual Features

In [33]:
class VisualEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = MobileNetV3Backbone()

        self.mdconv = DynamicMDConv(
            FEATURE_DIM
        )

        self.fpn = FeaturePyramidNetwork(
            FEATURE_DIM
        )

        self.attention = HybridAttention(
            FEATURE_DIM
        )

        self.projection = FeatureProjection()

        self.amsda = AMSDA()

    def forward(self, x):

        x = self.backbone(x)

        x = self.mdconv(x)

        x = self.fpn(x)

        x = self.attention(x)

        x = self.projection(x)

        x = self.amsda(x)

        return x

In [34]:
encoder = VisualEncoder().to(DEVICE)

dummy = torch.randn(
    2,
    3,
    224,
    224
).to(DEVICE)

encoder_output = encoder(dummy)

print(encoder_output.shape)

torch.Size([2, 256, 14, 14])


In [35]:
print("="*60)

print("Input Image :", dummy.shape)

print("Encoded Feature :", encoder_output.shape)

print("="*60)

Input Image : torch.Size([2, 3, 224, 224])
Encoded Feature : torch.Size([2, 256, 14, 14])


In [36]:
torch.save(
    encoder.state_dict(),
    "visual_encoder.pth"
)

print("Visual Encoder Saved Successfully!")

Visual Encoder Saved Successfully!


In [37]:
print("="*60)
print("NOTEBOOK 3 COMPLETED")
print("="*60)

print("✓ MobileNetV3 Backbone")
print("✓ Dynamic MDConv")
print("✓ Feature Pyramid Network")
print("✓ Hybrid Attention")
print("✓ Feature Projection")
print("✓ AMSDA")
print("✓ Complete Visual Encoder")

print("\nFinal Output Shape :", encoder_output.shape)

print("\nReady for Notebook 4 (Knowledge Engine)")

NOTEBOOK 3 COMPLETED
✓ MobileNetV3 Backbone
✓ Dynamic MDConv
✓ Feature Pyramid Network
✓ Hybrid Attention
✓ Feature Projection
✓ AMSDA
✓ Complete Visual Encoder

Final Output Shape : torch.Size([2, 256, 14, 14])

Ready for Notebook 4 (Knowledge Engine)
